# Обучение единой глобальной модели для прогнозирования балансов клиентов

В качестве модели у нас будет градиентный бустинг, а именно буду использовать `CatBoost`.

Модель будет принимать на вход горизонт и целевой месяц как фичи.

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

%cd ../..
import src.config as config

/home/financier/projects/bank-ts-pred


### 1. Загрузка и подготовка данных

Загружаем тот самый датасет в длинном формате, который мы подготовили. Далее я решил заполнить пропуски в лагах нулями, хотя до этого я такие строки удалял.

На это есть несколько причин. Возможно, заказчику нужно будет делать предсказания не только по клиентам, которые пользуются определенными продуктами в банке давно, поэтому нужно обучить модель предсказывать таргет по неполным историческим данным.

In [2]:
df = pd.read_parquet(config.SUPERVISED_HORIZON_PATH)
print(f"Размер датасета: {df.shape[0]:,} строк, {df.shape[1]} колонок")

lag_cols = [c for c in df.columns if any(x in c for x in ["lag_", "roll_", "trend_"])]
df[lag_cols] = df[lag_cols].fillna(0)

# на всякий случаем кастим эти признаки к str, чтобы CatBoost воспринимал их как категориальные признаки
cat_features = ["product", "gender", "segment", "region", "target_month"]
for col in cat_features:
    df[col] = df[col].astype(str)

Размер датасета: 27,700,983 строк, 29 колонок


In [3]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 27700983 entries, 0 to 27700982
Data columns (total 29 columns):
 #   Column         Dtype         
---  ------         -----         
 0   client_id      int64         
 1   report_date    datetime64[us]
 2   product        str           
 3   balance        float64       
 4   lag_1          float64       
 5   lag_2          float64       
 6   lag_3          float64       
 7   lag_6          float64       
 8   lag_12         float64       
 9   roll_mean_3    float64       
 10  roll_mean_6    float64       
 11  roll_mean_12   float64       
 12  roll_std_6     float64       
 13  roll_min_12    float64       
 14  roll_max_12    float64       
 15  trend_3        float64       
 16  month          int32         
 17  quarter        int32         
 18  is_december    int32         
 19  is_summer      int32         
 20  gender         str           
 21  age            int64         
 22  segment        str           
 23  region         s

### 2. Out-of-Time split:

Я зафиксировал тестовую точку отсчета - `2025-04-30`. Именно из этой точки мы будем делать предсказания на 12 месяцев вперед, то есть до конца доступной нам истории - `2026-04-30`.

Чтобы избежать утечку данных, нужно чтобы модель не заглядывала в будущее тестовой выборки, поэтому в train могут войти только те наблюдения, у который **дата таргета** меньше или же равна нашей выбранной точке - `2025-04-30`.

Для этого я решил перевести `report_date` и `horizon` в количество месяцев именно от начала нашей истории.

понял что проще по-другому и просто год умножить на 12 )))

In [4]:
SPLIT_DATE = pd.Timestamp('2025-04-30')

df['report_months_idx'] = df['report_date'].dt.year * 12 + df['report_date'].dt.month
df['target_months_idx'] = df['report_months_idx'] + df['horizon']

split_months_idx = SPLIT_DATE.year * 12 + SPLIT_DATE.month
train_mask = df['target_months_idx'] <= split_months_idx
test_mask = df['report_date'] == SPLIT_DATE

train_data = df[train_mask].copy()
test_data = df[test_mask].copy()

print(f"Обучающая выборка: {len(train_data):,}")
print(f"Тестовая выборка: {len(test_data):,}")

Обучающая выборка: 19,506,642
Тестовая выборка: 682,560


In [5]:
drop_cols = ['report_months_idx', 'target_months_idx']
train_data.drop(columns=drop_cols, inplace=True)
test_data.drop(columns=drop_cols, inplace=True)

### 3. Создаем матрицу $X$ и $y$

Для $X$ уберем идентификаторы и таргет.

In [6]:
drop_cols = ['client_id', 'report_date', 'target']

X_train = train_data.drop(columns=drop_cols)
y_train = train_data['target']

X_test = test_data.drop(columns=drop_cols)
y_test = test_data['target']

print(f"Сколько всего признаков вышло для обучения: {X_train.shape[1]}")

Сколько всего признаков вышло для обучения: 26


### 4. Наконец-то обучаем модель

In [ ]:
model = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    random_seed=config.SEED,
    verbose=100,
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    plot=True,
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 97755.9246628	test: 96643.1563887	best: 96643.1563887 (0)	total: 53.1s	remaining: 17h 40m 13s


### 5. Считаем метрики 

In [ ]:
def wape(y_true, y_pred):
    return (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100


test_data["preds"] = np.clip(model.predict(X_test), 0, None)
test_data["baseline_preds"] = test_data["balance"]


horizon_metrics = []

for h in sorted(test_data["horizon"].unique()):
    sub = test_data[test_data["horizon"] == h]

    mae_m = mean_absolute_error(sub["target"], sub["preds"])
    wape_m = wape(sub["target"], sub["preds"])

    mae_b = mean_absolute_error(sub["target"], sub["baseline_preds"])
    wape_b = wape(sub["target"], sub["baseline_preds"])

    horizon_metrics.append(
        {
            "Горизонт (мес)": h,
            "Модель MAE": round(mae_m, 2),
            "Бейзлайн MAE": round(mae_b, 2),
            "Модель WAPE %": round(wape_m, 3),
            "Бейзлайн WAPE %": round(wape_b, 3),
        }
    )